# **Dataset Penjualan (Dummy Data)**

Dataset yang digunakan pada tahap pengembangan awal merupakan data sintetis
(dummy data) yang dibuat berdasarkan alur bisnis dan struktur pembagian hasil
yang diperoleh dari hasil wawancara dengan UMKM.

Dataset ini tidak merepresentasikan transaksi aktual Arloraka Coffee dan hanya
digunakan untuk pengembangan, pengujian, dan demonstrasi prototype.

Aturan bisnis yang digunakan antara lain:
- pembagian hasil sebesar 10% untuk freelancer dan 90% untuk manajemen;
- harga menu mengikuti informasi yang diberikan UMKM;
- jumlah transaksi, frekuensi penjualan, lokasi, metode pembayaran, dan pola
  pembelian merupakan asumsi sintetis untuk keperluan simulasi.

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print(f"Random seed: {SEED}")

Random seed: 42


## **Jumlah Data dan Periode**

In [3]:
N_TRANSACTIONS = 750

START_DATE = pd.Timestamp("2026-07-01")
END_DATE = pd.Timestamp("2026-10-20")

print("Jumlah transaksi :", N_TRANSACTIONS)
print("Periode          :", START_DATE.date(), "-", END_DATE.date())

Jumlah transaksi : 750
Periode          : 2026-07-01 - 2026-10-20


## **Menu & Probability**

In [4]:
MENU_PRICES = {
    "americano": 10_000,
    "latte": 13_000,
    "aren_latte": 16_000,
    "matcha_latte": 16_000,
    "redvelvet_latte": 14_000,
    "mocha_latte": 14_000
}

menu_df = pd.DataFrame(
    list(MENU_PRICES.items()),
    columns=["menu", "harga"]
)

menu_df

,menu,harga
0,americano,10000
1,latte,13000
2,aren_latte,16000
3,matcha_latte,16000
4,redvelvet_latte,14000
5,mocha_latte,14000


In [5]:
MENU_NAMES = list(MENU_PRICES.keys())

MENU_PROBABILITIES = np.array([
    0.14,  # Americano
    0.16,  # Latte
    0.22,  # Aren Latte
    0.20,  # Matcha Latte
    0.14,  # Redvelvet Latte
    0.14   # Mocha Latte
])

assert np.isclose(MENU_PROBABILITIES.sum(), 1.0)

pd.DataFrame({
    "menu": MENU_NAMES,
    "probabilitas": MENU_PROBABILITIES
})

,menu,probabilitas
0,americano,0.14
1,latte,0.16
2,aren_latte,0.22
3,matcha_latte,0.20
4,redvelvet_latte,0.14
5,mocha_latte,0.14


## **Freelance Dummy**

In [6]:
freelancers = pd.DataFrame({
    "freelancer_id": [
        "FL-001", "FL-002", "FL-003", "FL-004", "FL-005",
        "FL-006", "FL-007", "FL-008", "FL-009", "FL-010"
    ],
    "nama_freelancer": [
        "Andi", "Bima", "Citra", "Dimas", "Eka",
        "Farhan", "Gita", "Hana", "Iqbal", "Jihan"
    ],
    "activity_weight": [
        0.15, 0.13, 0.12, 0.11, 0.10,
        0.10, 0.09, 0.08, 0.07, 0.05
    ]
})

assert np.isclose(freelancers["activity_weight"].sum(), 1.0)

freelancers

,freelancer_id,nama_freelancer,activity_weight
0,FL-001,Andi,0.15
1,FL-002,Bima,0.13
2,FL-003,Citra,0.12
3,FL-004,Dimas,0.11
4,FL-005,Eka,0.10
5,FL-006,Farhan,0.10
6,FL-007,Gita,0.09
7,FL-008,Hana,0.08
8,FL-009,Iqbal,0.07
9,FL-010,Jihan,0.05


### **Location**

In [7]:
LOCATIONS = [
    "Area Kampus",
    "Event Komunitas",
    "Area Perkantoran",
    "Car Free Day",
    "Acara Organisasi",
    "Penjualan Langsung"
]

LOCATION_PROBABILITIES = [
    0.28,
    0.18,
    0.16,
    0.15,
    0.12,
    0.11
]

pd.DataFrame({
    "lokasi": LOCATIONS,
    "probabilitas": LOCATION_PROBABILITIES
})

,lokasi,probabilitas
0,Area Kampus,0.28
1,Event Komunitas,0.18
2,Area Perkantoran,0.16
3,Car Free Day,0.15
4,Acara Organisasi,0.12
5,Penjualan Langsung,0.11


## **Payment Methode**

In [8]:
PAYMENT_METHODS = [
    "QRIS",
    "Transfer",
    "Cash"
]

PAYMENT_PROBABILITIES = [
    0.55,
    0.20,
    0.25
]

## **Date Time List**

In [9]:
date_range = pd.date_range(
    start=START_DATE,
    end=END_DATE,
    freq="D"
)

date_weights = []

for date in date_range:
    day = date.dayofweek

    # Senin = 0, Minggu = 6
    if day in [5, 6]:
        weight = 1.40
    elif day == 4:
        weight = 1.20
    else:
        weight = 1.00

    date_weights.append(weight)

date_weights = np.array(date_weights)
date_weights = date_weights / date_weights.sum()

print("Jumlah hari:", len(date_range))

Jumlah hari: 112


## **Transaction per Bottle**

In [10]:
def generate_total_bottles():
    values = [1, 2, 3, 4, 5, 6, 8, 10]

    probabilities = [
        0.25,
        0.27,
        0.20,
        0.12,
        0.07,
        0.05,
        0.025,
        0.015
    ]

    return np.random.choice(
        values,
        p=probabilities
    )

## **Flavour Composition**

In [11]:
def generate_menu_quantities(total_bottles):

    selected_menus = np.random.choice(
        MENU_NAMES,
        size=total_bottles,
        p=MENU_PROBABILITIES
    )

    quantities = {
        menu: 0
        for menu in MENU_NAMES
    }

    for menu in selected_menus:
        quantities[menu] += 1

    return quantities

## **Verification Status**

In [12]:
def generate_verification_status():
    statuses = [
        "Verified",
        "Pending",
        "Rejected"
    ]

    probabilities = [
        0.92,
        0.06,
        0.02
    ]

    return np.random.choice(
        statuses,
        p=probabilities
    )

In [13]:
def generate_note(location, total_bottles):

    if total_bottles >= 8:
        return "Pembelian dalam jumlah besar"

    if location == "Event Komunitas":
        return "Penjualan saat event komunitas"

    if location == "Car Free Day":
        return "Penjualan saat kegiatan CFD"

    if location == "Area Kampus":
        return "Penjualan area kampus"

    return "-"

## **Generate Transaction**

In [14]:
transactions = []

freelancer_probabilities = freelancers["activity_weight"].values

for i in range(1, N_TRANSACTIONS + 1):

    # Transaction ID
    transaction_id = f"TRX-{i:04d}"

    # Tanggal
    tanggal_jual = np.random.choice(
        date_range,
        p=date_weights
    )

    tanggal_jual = pd.Timestamp(tanggal_jual)

    # Freelancer
    freelancer_index = np.random.choice(
        freelancers.index,
        p=freelancer_probabilities
    )

    freelancer_id = freelancers.loc[
        freelancer_index,
        "freelancer_id"
    ]

    nama_freelancer = freelancers.loc[
        freelancer_index,
        "nama_freelancer"
    ]

    # Jumlah botol
    total_bottles = generate_total_bottles()

    # Komposisi menu
    quantities = generate_menu_quantities(
        total_bottles
    )

    # Hitung omzet
    omzet = sum(
        quantities[menu] * MENU_PRICES[menu]
        for menu in MENU_NAMES
    )

    # Pembagian hasil
    komisi_freelancer = int(round(omzet * 0.10))
    pendapatan_manajemen = omzet - komisi_freelancer

    # Pembayaran
    metode_pembayaran = np.random.choice(
        PAYMENT_METHODS,
        p=PAYMENT_PROBABILITIES
    )

    # Lokasi
    lokasi_penjualan = np.random.choice(
        LOCATIONS,
        p=LOCATION_PROBABILITIES
    )

    # Status laporan
    status_verifikasi = generate_verification_status()

    # Keterangan
    keterangan = generate_note(
        lokasi_penjualan,
        total_bottles
    )

    # Simpan row
    transactions.append({
        "transaction_id": transaction_id,
        "tanggal_jual": tanggal_jual,
        "freelancer_id": freelancer_id,
        "nama_freelancer": nama_freelancer,

        "americano_qty": quantities["americano"],
        "latte_qty": quantities["latte"],
        "aren_latte_qty": quantities["aren_latte"],
        "matcha_latte_qty": quantities["matcha_latte"],
        "redvelvet_latte_qty": quantities["redvelvet_latte"],
        "mocha_latte_qty": quantities["mocha_latte"],

        "total_botol": total_bottles,
        "omzet_penjualan": omzet,
        "komisi_freelancer": komisi_freelancer,
        "pendapatan_manajemen": pendapatan_manajemen,

        "metode_pembayaran": metode_pembayaran,
        "lokasi_penjualan": lokasi_penjualan,
        "status_verifikasi": status_verifikasi,
        "keterangan": keterangan
    })

In [15]:
df = pd.DataFrame(transactions)

df.head(10)

,transaction_id,tanggal_jual,freelancer_id,nama_freelancer,americano_qty,latte_qty,aren_latte_qty,matcha_latte_qty,redvelvet_latte_qty,mocha_latte_qty,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen,metode_pembayaran,lokasi_penjualan,status_verifikasi,keterangan
0,TRX-0001,2026-08-11,FL-010,Jihan,1,2,0,1,0,0,4,52000,5200,46800,Cash,Area Perkantoran,Verified,-
1,TRX-0002,2026-07-03,FL-010,Jihan,0,3,1,0,0,0,4,55000,5500,49500,QRIS,Event Komunitas,Verified,Penjualan saat event komunitas
2,TRX-0003,2026-09-07,FL-001,Andi,0,0,2,0,0,0,2,32000,3200,28800,Cash,Area Kampus,Verified,Penjualan area kampus
3,TRX-0004,2026-09-05,FL-001,Andi,1,1,0,0,0,1,3,37000,3700,33300,Cash,Acara Organisasi,Verified,-
4,TRX-0005,2026-07-11,FL-006,Farhan,1,0,1,0,0,0,2,26000,2600,23400,QRIS,Penjualan Langsung,Verified,-
5,TRX-0006,2026-09-13,FL-003,Citra,0,1,0,1,0,1,3,43000,4300,38700,Cash,Penjualan Langsung,Verified,-
6,TRX-0007,2026-09-05,FL-009,Iqbal,0,1,0,0,0,0,1,13000,1300,11700,QRIS,Event Komunitas,Verified,Penjualan saat event komunitas
7,TRX-0008,2026-07-31,FL-008,Hana,0,1,0,1,0,0,2,29000,2900,26100,QRIS,Acara Organisasi,Verified,-
8,TRX-0009,2026-10-19,FL-007,Gita,1,0,0,0,0,0,1,10000,1000,9000,Cash,Car Free Day,Verified,Penjualan saat kegiatan CFD
9,TRX-0010,2026-09-25,FL-001,Andi,1,0,0,0,0,1,2,24000,2400,21600,Transfer,Event Komunitas,Verified,Penjualan saat event komunitas


In [16]:
df = (
    df
    .sort_values(
        by=["tanggal_jual", "transaction_id"]
    )
    .reset_index(drop=True)
)

df.head()

,transaction_id,tanggal_jual,freelancer_id,nama_freelancer,americano_qty,latte_qty,aren_latte_qty,matcha_latte_qty,redvelvet_latte_qty,mocha_latte_qty,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen,metode_pembayaran,lokasi_penjualan,status_verifikasi,keterangan
0,TRX-0161,2026-07-01,FL-010,Jihan,0,0,2,0,0,1,3,46000,4600,41400,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD
1,TRX-0465,2026-07-01,FL-007,Gita,1,1,0,0,0,0,2,23000,2300,20700,QRIS,Acara Organisasi,Pending,-
2,TRX-0662,2026-07-01,FL-004,Dimas,0,1,1,0,1,0,3,43000,4300,38700,Cash,Area Perkantoran,Verified,-
3,TRX-0103,2026-07-02,FL-001,Andi,1,0,0,0,0,0,1,10000,1000,9000,QRIS,Area Perkantoran,Verified,-
4,TRX-0251,2026-07-02,FL-004,Dimas,0,1,0,1,0,0,2,29000,2900,26100,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD


In [17]:
print("Jumlah transaksi :", len(df))
print("Jumlah kolom      :", len(df.columns))

df.shape

Jumlah transaksi : 750
Jumlah kolom      : 18


(750, 18)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   transaction_id        750 non-null    object        
 1   tanggal_jual          750 non-null    datetime64[ns]
 2   freelancer_id         750 non-null    object        
 3   nama_freelancer       750 non-null    object        
 4   americano_qty         750 non-null    int64         
 5   latte_qty             750 non-null    int64         
 6   aren_latte_qty        750 non-null    int64         
 7   matcha_latte_qty      750 non-null    int64         
 8   redvelvet_latte_qty   750 non-null    int64         
 9   mocha_latte_qty       750 non-null    int64         
 10  total_botol           750 non-null    int64         
 11  omzet_penjualan       750 non-null    int64         
 12  komisi_freelancer     750 non-null    int64         
 13  pendapatan_manajemen

## **Validating Bottle Total**

In [19]:
menu_qty_columns = [
    "americano_qty",
    "latte_qty",
    "aren_latte_qty",
    "matcha_latte_qty",
    "redvelvet_latte_qty",
    "mocha_latte_qty"
]

df["calculated_total_botol"] = (
    df[menu_qty_columns]
    .sum(axis=1)
)

invalid_total = df[
    df["calculated_total_botol"]
    != df["total_botol"]
]

print("Data dengan total botol tidak valid:", len(invalid_total))

invalid_total.head()

Data dengan total botol tidak valid: 0


,transaction_id,tanggal_jual,freelancer_id,nama_freelancer,americano_qty,latte_qty,aren_latte_qty,matcha_latte_qty,redvelvet_latte_qty,mocha_latte_qty,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen,metode_pembayaran,lokasi_penjualan,status_verifikasi,keterangan,calculated_total_botol


In [20]:
df.drop(
    columns=["calculated_total_botol"],
    inplace=True
)

## **Omzet**

In [21]:
df["calculated_omzet"] = (
    df["americano_qty"] * MENU_PRICES["americano"]
    + df["latte_qty"] * MENU_PRICES["latte"]
    + df["aren_latte_qty"] * MENU_PRICES["aren_latte"]
    + df["matcha_latte_qty"] * MENU_PRICES["matcha_latte"]
    + df["redvelvet_latte_qty"] * MENU_PRICES["redvelvet_latte"]
    + df["mocha_latte_qty"] * MENU_PRICES["mocha_latte"]
)

invalid_omzet = df[
    df["calculated_omzet"]
    != df["omzet_penjualan"]
]

print("Data omzet tidak valid:", len(invalid_omzet))

Data omzet tidak valid: 0


In [22]:
df.drop(
    columns=["calculated_omzet"],
    inplace=True
)

## **Validating Konsinyasi**

In [23]:
check_split = (
    df["komisi_freelancer"]
    + df["pendapatan_manajemen"]
)

invalid_split = df[
    check_split != df["omzet_penjualan"]
]

print(
    "Data pembagian hasil tidak valid:",
    len(invalid_split)
)

Data pembagian hasil tidak valid: 0


## **Missing Value**

In [24]:
df.isnull().sum()

,0
transaction_id,0
tanggal_jual,0
freelancer_id,0
nama_freelancer,0
americano_qty,0
latte_qty,0
aren_latte_qty,0
matcha_latte_qty,0
redvelvet_latte_qty,0
mocha_latte_qty,0


## **Basic Dataset**

In [25]:
df[
    [
        "total_botol",
        "omzet_penjualan",
        "komisi_freelancer",
        "pendapatan_manajemen"
    ]
].describe()

,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen
count,750.000000,750.000000,750.000000,750.000000
mean,2.860000,40025.333333,4002.533333,36022.800000
std,1.937715,27541.606313,2754.160631,24787.445682
min,1.000000,10000.000000,1000.000000,9000.000000
25%,1.000000,16000.000000,1600.000000,14400.000000
50%,2.000000,32000.000000,3200.000000,28800.000000
75%,4.000000,52000.000000,5200.000000,46800.000000
max,10.000000,153000.000000,15300.000000,137700.000000


Ringkasan Total Transaction

In [26]:
summary = {
    "jumlah_transaksi": len(df),
    "total_botol_terjual": df["total_botol"].sum(),
    "total_omzet": df["omzet_penjualan"].sum(),
    "total_komisi_freelancer": df["komisi_freelancer"].sum(),
    "total_pendapatan_manajemen": df["pendapatan_manajemen"].sum()
}

summary

{'jumlah_transaksi': 750,
 'total_botol_terjual': np.int64(2145),
 'total_omzet': np.int64(30019000),
 'total_komisi_freelancer': np.int64(3001900),
 'total_pendapatan_manajemen': np.int64(27017100)}

Penjualan per Menu

In [27]:
sales_by_menu = pd.DataFrame({
    "menu": [
        "Americano",
        "Latte",
        "Aren Latte",
        "Matcha Latte",
        "Redvelvet Latte",
        "Mocha Latte"
    ],

    "jumlah_terjual": [
        df["americano_qty"].sum(),
        df["latte_qty"].sum(),
        df["aren_latte_qty"].sum(),
        df["matcha_latte_qty"].sum(),
        df["redvelvet_latte_qty"].sum(),
        df["mocha_latte_qty"].sum()
    ]
})

sales_by_menu.sort_values(
    "jumlah_terjual",
    ascending=False
)

,menu,jumlah_terjual
2,Aren Latte,444
3,Matcha Latte,401
1,Latte,381
0,Americano,330
5,Mocha Latte,299
4,Redvelvet Latte,290


Performa Freelancer


In [28]:
freelancer_performance = (
    df
    .groupby(
        ["freelancer_id", "nama_freelancer"],
        as_index=False
    )
    .agg(
        jumlah_transaksi=("transaction_id", "count"),
        total_botol=("total_botol", "sum"),
        omzet_penjualan=("omzet_penjualan", "sum"),
        komisi_freelancer=("komisi_freelancer", "sum")
    )
    .sort_values(
        "omzet_penjualan",
        ascending=False
    )
)

freelancer_performance

,freelancer_id,nama_freelancer,jumlah_transaksi,total_botol,omzet_penjualan,komisi_freelancer
0,FL-001,Andi,117,321,4454000,445400
3,FL-004,Dimas,91,262,3674000,367400
1,FL-002,Bima,88,257,3554000,355400
4,FL-005,Eka,81,243,3414000,341400
2,FL-003,Citra,87,243,3387000,338700
7,FL-008,Hana,61,209,2955000,295500
6,FL-007,Gita,76,202,2846000,284600
5,FL-006,Farhan,62,166,2306000,230600
9,FL-010,Jihan,47,126,1793000,179300
8,FL-009,Iqbal,40,116,1636000,163600


Penjualan per Tanggal

In [29]:
daily_sales = (
    df
    .groupby(
        "tanggal_jual",
        as_index=False
    )
    .agg(
        jumlah_transaksi=("transaction_id", "count"),
        total_botol=("total_botol", "sum"),
        omzet=("omzet_penjualan", "sum")
    )
)

daily_sales.head()

,tanggal_jual,jumlah_transaksi,total_botol,omzet
0,2026-07-01,3,8,112000
1,2026-07-02,6,13,177000
2,2026-07-03,9,37,509000
3,2026-07-04,8,26,378000
4,2026-07-05,9,24,335000


Penjualan berdasarkan Lokasi

In [30]:
location_performance = (
    df
    .groupby(
        "lokasi_penjualan",
        as_index=False
    )
    .agg(
        jumlah_transaksi=("transaction_id", "count"),
        total_botol=("total_botol", "sum"),
        omzet=("omzet_penjualan", "sum")
    )
    .sort_values(
        "omzet",
        ascending=False
    )
)

location_performance

,lokasi_penjualan,jumlah_transaksi,total_botol,omzet
1,Area Kampus,205,591,8337000
3,Car Free Day,134,406,5644000
2,Area Perkantoran,126,367,5083000
4,Event Komunitas,101,281,3943000
0,Acara Organisasi,96,281,3906000
5,Penjualan Langsung,88,219,3106000


Metode Pembayaran

In [31]:
df["metode_pembayaran"].value_counts()

,count
metode_pembayaran,
QRIS,418
Cash,170
Transfer,162


Status Verifikasi

In [32]:
df["status_verifikasi"].value_counts()

,count
status_verifikasi,
Verified,693
Pending,42
Rejected,15


In [33]:
df.head(20)

,transaction_id,tanggal_jual,freelancer_id,nama_freelancer,americano_qty,latte_qty,aren_latte_qty,matcha_latte_qty,redvelvet_latte_qty,mocha_latte_qty,total_botol,omzet_penjualan,komisi_freelancer,pendapatan_manajemen,metode_pembayaran,lokasi_penjualan,status_verifikasi,keterangan
0,TRX-0161,2026-07-01,FL-010,Jihan,0,0,2,0,0,1,3,46000,4600,41400,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD
1,TRX-0465,2026-07-01,FL-007,Gita,1,1,0,0,0,0,2,23000,2300,20700,QRIS,Acara Organisasi,Pending,-
2,TRX-0662,2026-07-01,FL-004,Dimas,0,1,1,0,1,0,3,43000,4300,38700,Cash,Area Perkantoran,Verified,-
3,TRX-0103,2026-07-02,FL-001,Andi,1,0,0,0,0,0,1,10000,1000,9000,QRIS,Area Perkantoran,Verified,-
4,TRX-0251,2026-07-02,FL-004,Dimas,0,1,0,1,0,0,2,29000,2900,26100,QRIS,Car Free Day,Verified,Penjualan saat kegiatan CFD
5,TRX-0308,2026-07-02,FL-006,Farhan,1,0,0,0,0,1,2,24000,2400,21600,QRIS,Area Kampus,Verified,Penjualan area kampus
6,TRX-0443,2026-07-02,FL-007,Gita,1,0,0,1,0,0,2,26000,2600,23400,QRIS,Penjualan Langsung,Verified,-
7,TRX-0606,2026-07-02,FL-006,Farhan,0,0,1,0,0,1,2,30000,3000,27000,Transfer,Acara Organisasi,Verified,-
8,TRX-0735,2026-07-02,FL-001,Andi,0,0,0,1,2,1,4,58000,5800,52200,QRIS,Acara Organisasi,Verified,-
9,TRX-0002,2026-07-03,FL-010,Jihan,0,3,1,0,0,0,4,55000,5500,49500,QRIS,Event Komunitas,Verified,Penjualan saat event komunitas


Final Dataset Master

In [34]:
CSV_FILE = "dummy_sales_arloraka_coffee_750.csv"

df.to_csv(
    CSV_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Dataset berhasil disimpan: {CSV_FILE}"
)

Dataset berhasil disimpan: dummy_sales_arloraka_coffee_750.csv


In [35]:
EXCEL_FILE = "dummy_sales_arloraka_coffee_750.xlsx"

df.to_excel(
    EXCEL_FILE,
    index=False
)

print(
    f"Dataset berhasil disimpan: {EXCEL_FILE}"
)

Dataset berhasil disimpan: dummy_sales_arloraka_coffee_750.xlsx


In [36]:
assert len(df) == 750

assert df["transaction_id"].nunique() == 750

assert (
    df[menu_qty_columns].sum(axis=1)
    == df["total_botol"]
).all()

assert (
    df["komisi_freelancer"]
    + df["pendapatan_manajemen"]
    == df["omzet_penjualan"]
).all()

assert not df.isnull().any().any()

print("SEMUA VALIDASI BERHASIL")
print("Jumlah transaksi :", len(df))
print("Total botol      :", df["total_botol"].sum())
print("Total omzet      :", f"Rp{df['omzet_penjualan'].sum():,}")
print("Jumlah freelancer:", df["freelancer_id"].nunique())

SEMUA VALIDASI BERHASIL
Jumlah transaksi : 750
Total botol      : 2145
Total omzet      : Rp30,019,000
Jumlah freelancer: 10
